# 04 — Ingeniería de características

Generar las nuevas variables matemáticas que capturan la evolución temporal del alumno (delta, brecha, ritmo, años sin finales).
Entender qué mide cada feature y dejar el dataset final listo para modelar.
Ejecutar el test de fuego anti-leakage con pytest para garantizar que ninguna variable use información del futuro.

Unidad de análisis: un estudiante en un año académico.

In [1]:
import sys
import os
from pathlib import Path

PROJ_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJ_ROOT)
if str(PROJ_ROOT) not in sys.path:
    sys.path.append(str(PROJ_ROOT))

import pandas as pd
from src.features.build_features import agregar_features, FEATURES_COLS, matriz_modelado

# Cargar el dataset con target que guardamos en la Semana 2
df = pd.read_parquet("data/processed/dataset_modelado.parquet")
print(f"Dataset cargado: {df.shape[0]} filas y {df.shape[1]} columnas.")


Dataset cargado: 1979 filas y 36 columnas.


In [2]:
df_features = agregar_features(df)
print(f"Dimensiones tras agregar features: {df_features.shape}")

# Inspeccionar las 4 features clave recién creadas
nuevas_cols = ["id_alumno", "anio_academico", "delta_tasa_aprobacion", 
              "brecha_cursado_carrera", "ritmo_vs_plan", "anios_sin_finales"]
df_features[nuevas_cols].head(10)

Dimensiones tras agregar features: (1979, 40)


/Users/alejo/Desktop/alerta-temprana-desercion/src/features/build_features.py:46: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["ritmo_vs_plan"] = (df["porcentaje_avance_carrera"] / ritmo_teorico).fillna(0)


,id_alumno,anio_academico,delta_tasa_aprobacion,brecha_cursado_carrera,ritmo_vs_plan,anios_sin_finales
0,2,2023,0.00,0.000,0.000,1
1,3,2003,0.00,0.013,0.000,1
2,3,2004,-0.33,0.013,0.364,2
3,3,2005,0.00,-0.010,0.228,0
4,4,2008,0.00,0.000,0.000,1
5,5,2021,0.00,0.000,0.000,1
6,6,2021,0.00,0.002,0.000,0
7,6,2022,-0.33,-0.005,1.272,0
8,6,2023,0.33,-0.008,0.682,0
9,7,2021,0.00,-0.002,0.000,0


*Regla anti-leakage:** feature del año `t` solo usa info `≤ t`. El target mira `t+1, t+2`. Sin esto, el modelo no sirve en producción.

## Rendimiento académico
- `tasa_aprobacion_asignaturas` — % aprobadas. Predictor #1 de abandono.
- `tasa_desaprobacion_asignaturas` — aprueba fácil / reprueba difícil ≠ rendimiento uniforme.
- `tasa_ausencias_finales` — ausencia al final = desenganche, señal temprana.

## Avance y progreso
- `porcentaje_avance_carrera` — distancia al título (abandono pico a mitad de carrera).
- `porcentaje_avance_cursado` — cursar sin rendir ≠ no cursar.
- `antiguedad_carrera` — rezago crónico (3er año en 6 de carrera).
- `brecha_cursado_carrera` — mucho cursado + poco avance = materias pendientes acumuladas.
- `ritmo_vs_plan` — avance / (antigüedad/4). 1.0 = ritmo ideal. Comparable entre cohortes.

## Interrupciones
- `cantidad_pausas` — cada pausa previa ↑ probabilidad de la próxima.
- `maxima_duracion_pausa` — 1 año de pausa ≠ 4; mide gravedad.

## Dinámica (tendencia > nivel)
- `delta_tasa_aprobacion` — 85%→45% en caída importa más que 70% estable.
- `anios_sin_finales` — racha **consecutiva** actual, no acumulado (mide el presente).

## Perfil
- `edad` — incongruencia de edad con la cohorte predice abandono temprano.

## Target
- `abandono =` sin actividad en los 2 años siguientes a `t`.
- **Censura:** cohortes sin ventana completa → excluidas (`pd.NA`), no "abandono".

## Validación
1. Correlación feature vs target.
2. Importancia por permutación (caída de AUC).
3. Ablation de features dinámicas vs estáticas.
4. CV con `groups=id_alumno` (evita fuga entre folds).

**Resumen:** 13 features, 5 dimensiones ortogonales (rendimiento, avance, interrupciones, dinámica, perfil), sin redundancia y desplegables en producción.

In [3]:
df_features[["delta_tasa_aprobacion", "brecha_cursado_carrera", "ritmo_vs_plan", "anios_sin_finales"]].describe()

,delta_tasa_aprobacion,brecha_cursado_carrera,ritmo_vs_plan,anios_sin_finales
count,1979.000000,1979.000000,1979.000000,1979.000000
mean,-0.058247,0.001612,0.320899,0.466902
std,0.281419,0.031219,0.487003,0.747284
min,-1.000000,-0.112000,0.000000,0.000000
25%,0.000000,-0.015000,0.000000,0.000000
50%,0.000000,-0.002000,0.000000,0.000000
75%,0.000000,0.013000,0.500000,1.000000
max,1.000000,0.201000,2.820000,6.000000


In [4]:
X, y, groups = matriz_modelado(df_features)

print("Matriz X (predictoras):", X.shape)
print("Target y:", y.shape)
print("Grupos (id_alumno para GroupKFold):", groups.nunique(), "alumnos únicos")
print("\nLista final de features del modelo:")
for col in X.columns:
    print(f" - {col}")

Matriz X (predictoras): (1979, 13)
Target y: (1979,)
Grupos (id_alumno para GroupKFold): 1019 alumnos únicos

Lista final de features del modelo:
 - tasa_aprobacion_asignaturas
 - tasa_desaprobacion_asignaturas
 - tasa_ausencias_finales
 - porcentaje_avance_carrera
 - porcentaje_avance_cursado
 - cantidad_pausas
 - maxima_duracion_pausa
 - antiguedad_carrera
 - edad
 - delta_tasa_aprobacion
 - brecha_cursado_carrera
 - ritmo_vs_plan
 - anios_sin_finales


In [5]:
df_features.to_parquet("data/processed/dataset_modelado.parquet", index=False)
print("Dataset final guardado exitosamente.")

Dataset final guardado exitosamente.
